# ROBER: direct emulator
(initial state, t) -> y(t), fixed rate constants. Logic lives in `src/chemistry_emulator`; this notebook only runs it.

## Colab setup

In [ ]:
%rm -r chemistry-emulator
!git clone https://github.com/plienta/chemistry-emulator.git
%cd chemistry-emulator


In [ ]:
import sys
sys.path.append('/content/chemistry-emulator')

In [ ]:
!pip install -e /content/chemistry-emulator

In [ ]:
path = '/content/rober_data.npz'

## General config

In [ ]:
N_TRAJ, N_TIMES, EPOCHS = (2000, 50, 150)
N_WORKERS = 1

In [ ]:
from chemistry_emulator.systems.training import prepare, train, save_run
from chemistry_emulator.model.mlp import MLP
from chemistry_emulator.model.deep_o_net import DeepONet
from chemistry_emulator.systems.evaluate import evaluate_direct

## 1. Generate data

In [ ]:
import numpy as np
import os
from chemistry_emulator.robertson.rober import make_direct_data

In [ ]:
if os.path.exists(path):
    d = np.load(path); X, Y, TID = d["X"], d["Y"], d["TID"]
else:
    X, Y, TID = make_direct_data(N_TRAJ, N_TIMES, seed=0, n_workers=N_WORKERS)
    np.savez(path, X=X, Y=Y, TID=TID)

## 2. Split, normalise, train

In [ ]:
data = prepare(X, Y, TID, seed=0)
models = {"mlp": MLP(n_hidden=5, n_in=4, n_out=3),
          "deeponet": DeepONet(n_hidden=5, n_out=3, n_branch_in=3)}
results = {}

In [ ]:
for name, m in models.items():
    p = train(m, data, epochs=EPOCHS)
    results[name] = p
    save_run(f"/content/rober_direct_{name}.pkl", p, data)
    evaluate_direct(m, p, data)